In [ ]:
!pip install memory_profiler

In [ ]:
#select python kernel
from sage.all import *
import cplex

import time
import psutil
import os
import tracemalloc


# wall time, cpu, mem
def run_and_measure_once_py(f, *args, **kw):
    p = psutil.Process(os.getpid())

    cpu0 = p.cpu_times()
    t0 = time.perf_counter()
    tracemalloc.start()

    out = f(*args, **kw)

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    t1 = time.perf_counter()
    cpu1 = p.cpu_times()

    return {
        "result": out,
        "wall_time_s": t1 - t0,
        "cpu_time_s": (cpu1.user - cpu0.user) + (cpu1.system - cpu0.system),
        "peak_mem_MiB": peak / (1024 ** 2),
    }


# reading p list from file
def read_p(file):
    lista = []
    with open(file, 'r') as f:
        for line in f:
            s = line.strip()
            inner = s[1:-1].strip()
            sublist = [int(elem) for elem in inner.split(',') if elem.strip() != ""]
            lista.append(sublist)
    return lista


def hamming_weight(lst):
    return int(sum(lst))


# padding with zero s.t. the list of coeffs for q will be of length t+1 (maximum degree t)
def pad_with_zeros(P_coeffs, target_len):
    if len(P_coeffs) > target_len + 1:
        raise ValueError("List is longer than target_len")
    return P_coeffs + [0] * (target_len + 1 - len(P_coeffs))


def coeffs_to_sage_poly(P_coeffs, t):
    R = PolynomialRing(GF(2), 'x')
    x = R.gen()

    P_coeffs = pad_with_zeros(P_coeffs, t)
    P = sum(int(coef) * x**i for i, coef in enumerate(P_coeffs))

    return P


def find_multiple(P, d, w):
    d = int(d)
    w = int(w)

    R = P.parent()
    x = R.gen()

    t = int(P.degree())
    N = int(t + d)

    p = [int(P[i]) for i in range(t + 1)]

    model = cplex.Cplex()
    model.objective.set_sense(model.objective.sense.minimize)
    model.parameters.mip.limits.solutions.set(1)

    q = [f"q_{j}" for j in range(d + 1)]
    pq = [f"pq_{k}" for k in range(N + 1)]
    xor = [f"xor_{k}" for k in range(N + 1)]

    model.variables.add(names=q, lb=[0.0]*(d+1), ub=[1.0]*(d+1), types=["B"]*(d+1), obj=[0.0]*(d+1))
    model.variables.add(names=pq, lb=[0.0]*(N+1), ub=[1.0]*(N+1), types=["B"]*(N+1), obj=[0.0]*(N+1))
    model.variables.add(names=xor, lb=[0.0]*(N+1), ub=[cplex.infinity]*(N+1), types=["I"]*(N+1), obj=[0.0]*(N+1))


    for k in range(N + 1):
        vars_k = []
        coeffs_k = []

        for i in range(t + 1):
            j = k - i
            if 0 <= j <= d and p[i] == 1:
                vars_k.append(q[j])
                coeffs_k.append(1.0)

        vars_k += [pq[k], xor[k]]
        coeffs_k += [-1.0, -2.0]

        model.linear_constraints.add(
            lin_expr=[[vars_k, coeffs_k]],
            senses=["E"],
            rhs=[0.0]
        )

    # deg(Q) = d
    model.linear_constraints.add(
        lin_expr=[[[q[d]], [1.0]]],
        senses=["E"],
        rhs=[1.0]
    )

    # (w-1)//2 <= wt(PQ) <= w
    model.linear_constraints.add(
        lin_expr=[[pq, [1.0]*(N+1)]],
        senses=["G"],
        rhs=[float((w - 1)//2)]
    )

    
    model.linear_constraints.add(
        lin_expr=[[pq, [1.0]*(N+1)]],
        senses=["L"],
        rhs=[float(w)]
    )
    
    model.solve()

    status = model.solution.get_status_string()

    if not model.solution.is_primal_feasible():
        return P, None, None, None, status

    q_val = model.solution.get_values(q)
    pq_val = model.solution.get_values(pq)

    Q = sum(int(round(q_val[j])) * x**j for j in range(d + 1))
    PQ = sum(int(round(pq_val[k])) * x**k for k in range(N + 1))

    wt = PQ.hamming_weight()
    ok = (P * Q == PQ)

    return P, Q, PQ, wt, status if ok else "invalid"




In [ ]:

t = 400  # Degree of P
d = 100  # Degree of Q
w = 10 # hamming weight for P*Q

print("t,d,w =", t, d, w)

#path for input
lista_p = read_p("")

q_out = []
pq_out = []
weight_out = []
peakmem = []
cpu = []
wall = []
optimum = []

for poz, pcoeffs in enumerate(lista_p):
    print("p[{}] =".format(poz), pcoeffs)

    P = coeffs_to_sage_poly(pcoeffs, t)

    try:
        stats = run_and_measure_once_py(find_multiple, P=P, d=d, w=w)
    except Exception as e:
        print(f"Solver error at instance {poz}: {type(e).__name__}: {e}")
        continue

    P, Q, PQ, wt, optim = stats["result"]
    
    print("Q =", Q)
    print("weight(PQ) =", wt)
    print("status =", optim)
    print("==============================================================================================")

    if Q is None:
        q_out.append(None)
        pq_out.append(None)
        weight_out.append(None)
        optimum.append(optim)
    else:
        Q_vec = [int(c) for c in Q.list()]
        PQ_vec = [int(c) for c in PQ.list()]

        q_out.append(Q_vec)
        pq_out.append(PQ_vec)
        weight_out.append(wt)
        optimum.append(optim)

    peakmem.append(stats["peak_mem_MiB"])
    wall.append(stats["wall_time_s"])
    cpu.append(stats["cpu_time_s"])


# path for output
outdir = ""

os.makedirs(outdir, exist_ok=True)

with open(os.path.join(outdir, "result_q.txt"), "w") as f: 
    for elem in q_out: 
        f.write(f"{elem}\n")

with open(os.path.join(outdir, "result_norm.txt"), "w") as f: 
    for elem in weight_out:
        f.write(f"{elem}\n")

with open(os.path.join(outdir, "result_pq.txt"), "w") as f:
    for elem in pq_out:
        f.write(f"{elem}\n")

with open(os.path.join(outdir, "wall_time.txt"), "w") as f:
    for elem in wall:
        f.write(f"{elem}\n")

with open(os.path.join(outdir, "cpu_time.txt"), "w") as f:
    for elem in cpu:
        f.write(f"{elem}\n")

with open(os.path.join(outdir, "peak_mem.txt"), "w") as f:
    for elem in peakmem:
        f.write(f"{elem}\n") 

with open(os.path.join(outdir, "optim.txt"), "w") as f:
    for elem in optimum: 
        f.write(f"{elem}\n")

print("---------------------------------")
print("weight =", weight_out)
print("q =", q_out)